<a href="https://colab.research.google.com/github/Krishishah7/nlp-learning-series/blob/main/01_embeddings/word_vs_sentence_embeddings_comparison.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
!pip install gensim sentence-transformers scikit-learn

In [10]:
from gensim.models import Word2Vec
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

In [11]:
sentences = [
    "Machine learning improves decision making",
    "Deep learning is a part of machine learning",
    "Natural language processing works with text",
    "Football is a popular sport",
    "Cricket is widely played in India"
]

In [12]:
tokenized_sentences = [sentence.lower().split() for sentence in sentences]
tokenized_sentences

[['machine', 'learning', 'improves', 'decision', 'making'],
 ['deep', 'learning', 'is', 'a', 'part', 'of', 'machine', 'learning'],
 ['natural', 'language', 'processing', 'works', 'with', 'text'],
 ['football', 'is', 'a', 'popular', 'sport'],
 ['cricket', 'is', 'widely', 'played', 'in', 'india']]

In [13]:
word2vec_model = Word2Vec(
    sentences=tokenized_sentences,
    vector_size=100,
    window=5,
    min_count=1,
    workers=4
)

In [14]:
def sentence_embedding_word2vec(sentence, model):
    words = sentence.lower().split()
    vectors = [model.wv[word] for word in words if word in model.wv]
    return np.mean(vectors, axis=0)

word_sentence_embeddings = np.array(
    [sentence_embedding_word2vec(sentence, word2vec_model) for sentence in sentences]
)

word_sentence_embeddings.shape

(5, 100)

In [15]:
sentence_model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [16]:
sentence_embeddings = sentence_model.encode(sentences)
sentence_embeddings.shape

(5, 384)

In [17]:
query = "AI and machine learning techniques"

In [18]:
query_word_embedding = sentence_embedding_word2vec(query, word2vec_model)

word_similarities = cosine_similarity(
    [query_word_embedding],
    word_sentence_embeddings
)[0]

word_similarities

array([ 0.65185535,  0.65097624,  0.09011704, -0.14259546, -0.02632034],
      dtype=float32)

In [19]:
query_sentence_embedding = sentence_model.encode([query])

sentence_similarities = cosine_similarity(
    query_sentence_embedding,
    sentence_embeddings
)[0]

sentence_similarities

array([0.5012369 , 0.4501359 , 0.3013789 , 0.08356661, 0.12601122],
      dtype=float32)

In [20]:
print("Query:", query)
print("\n--- Word Embedding Similarity ---")
for s, score in zip(sentences, word_similarities):
    print(f"{score:.3f} → {s}")

print("\n--- Sentence Embedding Similarity ---")
for s, score in zip(sentences, sentence_similarities):
    print(f"{score:.3f} → {s}")

Query: AI and machine learning techniques

--- Word Embedding Similarity ---
0.652 → Machine learning improves decision making
0.651 → Deep learning is a part of machine learning
0.090 → Natural language processing works with text
-0.143 → Football is a popular sport
-0.026 → Cricket is widely played in India

--- Sentence Embedding Similarity ---
0.501 → Machine learning improves decision making
0.450 → Deep learning is a part of machine learning
0.301 → Natural language processing works with text
0.084 → Football is a popular sport
0.126 → Cricket is widely played in India


- This notebook compares word embeddings and sentence embeddings for semantic similarity tasks.

- Word embeddings represent individual words and require manual aggregation to represent sentences,
which often loses context and meaning.

- Sentence embeddings directly encode the entire sentence semantics, making them more effective
for similarity, search, and retrieval-based NLP applications.